In [ ]:
# Importing the libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# data import
data = pd.read_csv('/content/drive/MyDrive/Datasets/Input/E-com_Data.csv')

# data dimension
data.shape

In [ ]:
# data sample
data.head()

In [ ]:
!pip install ydata-profiling

In [ ]:
# Importing the library
from ydata_profiling import ProfileReport

# creating the report
profile = ProfileReport(data, title="Profiling Report")
profile.to_file(output_file = '/content/eda_output.html')

1. Recency: How recently a customer has made a purchase
2. Frequency: How often a customer makes a purchase
3. Monetary value: How much money a customer spends on purchases

In [ ]:
# data sample
data.head()

In [ ]:
# Columns used for RFM Analysis grouped by CustomerId
    # 1. Recency - Date of purchase
    # 2. Frequency - InvoiceNo
    # 3. Monetary - Price

In [ ]:
# retaining the columns needed for Analysis
data = data[['CustomerID', 'InvoieNo', 'Date of purchase', 'Price']]

# renaming the columns
data = data.rename(columns={'InvoieNo': 'InvoiceNo', 'Date of purchase': 'Date'})

# data sample after filterning and renaming
data.head()

In [ ]:
# missing value percentage
round(data.isnull().mean()*100, 2)

In [ ]:
# drop the missing value
data = data.dropna(subset=['CustomerID'])

# missing value percentage after treatment
round(data.isnull().mean()*100, 2)

In [ ]:
# data duplicates
print('Number of duplicate rows in the data before treating:', data.duplicated().sum())
data = data.drop_duplicates(ignore_index=True)
print('Number of duplicate rows in the data after treating:', data.duplicated().sum())

In [ ]:
# data type the data
data.dtypes

In [ ]:
# Typecasting CustomerID
data['CustomerID'] = data['CustomerID'].astype(int)
data['InvoiceNo'] = data['InvoiceNo'].astype(int)
data['Date'] = pd.to_datetime(data['Date'])


# data type after typecasting
data.dtypes

In [ ]:
# data sample
data.head()

In [ ]:
# Maximum date of purchase
data['Date'].max()

In [ ]:
# latest date of purchase
latest_date = dt.datetime(2017, 12, 20)
latest_date

In [ ]:
# RFM Function
RFMScore = data.groupby('CustomerID').agg({'Date': lambda x: (latest_date-x.max()).days,  # Recency
                                           'InvoiceNo': lambda x: x.count(),              # Frequency
                                           'Price': lambda x: x.sum()})                   # Monetory

# Renaming the columns
RFMScore.rename(columns={'Date': 'Recency', 'InvoiceNo': 'Frequency', 'Price': 'Monetory'}, inplace=True)

# converting the data to dataframe
RFMScore.reset_index().head()

In [ ]:
# minimum and maximum values of each attributes
for cols in RFMScore.columns:
    print(f'For {cols} the min value is {RFMScore[cols].min()} and max value is {RFMScore[cols].max()}.')

In [ ]:
# Quantile or splitting the data
quantile = RFMScore.quantile(q=[0.25, 0.5, 0.75])
quantile = quantile.to_dict()
quantile

In [ ]:
# Function for the score

# lower the value of recency more valuable the customer is

def recency_score(x, q, d):
    if x <= d[q][0.25]:
        return 1            # higher rank
    elif x <= d[q][0.50]:
        return 2
    elif x <= d[q][0.75]:
        return 3
    else:
        return 4            # lower rank

# higher the value of frequency & monetory more valuable the customer is

def FnM_Score(x, q, d):
    if x <= d[q][0.25]:     # lower rank
        return 4
    elif x <= d[q][0.50]:
        return 3
    elif x <= d[q][0.75]:
        return 2
    else:
        return 1            # higher rank

In [ ]:
# columns to accomodate the scores from the function
RFMScore['R'] = RFMScore['Recency'].apply(recency_score, args=('Recency', quantile, ))
RFMScore['F'] = RFMScore['Frequency'].apply(FnM_Score, args=('Frequency', quantile, ))
RFMScore['M'] = RFMScore['Monetory'].apply(FnM_Score, args=('Monetory', quantile, ))

In [ ]:
# RFM data
RFMScore.head()

In [ ]:
# Loyality Score
RFMScore['LoyalityScore'] = RFMScore[['R', 'F', 'M']].sum(axis=1)
RFMScore.head()

In [ ]:
# Loyality Badge
badge = ['Platinum', 'Gold', 'Silver', 'Bronze']
score_cut = pd.qcut(RFMScore.LoyalityScore, 4, labels=badge)
RFMScore['LoyalityBadge'] = score_cut.values
RFMScore.head()

In [ ]:
# Segmeneted data
segemented_data = RFMScore.reset_index()
segemented_data = segemented_data[['CustomerID', 'Recency', 'Frequency', 'Monetory', 'LoyalityBadge']]

# mapped data sample
segemented_data.head()

In [ ]:
# Distribution of the customers
ax = sns.countplot(x=segemented_data['LoyalityBadge'],
              order=segemented_data['LoyalityBadge'].value_counts().index)
ax.bar_label(ax.containers[0])
plt.show()